# Minimal GPU + GM/Redi passive-dye reproducer

This notebook directly addresses: **‘the DomainError may be worth an issue, if you can come up with a small reproducer.’**

It removes NumericalEarth, ECCO, JRA55, rivers, sponges, output writers, and plotting. The only retained ingredients are a tiny hydrostatic model, nonnegative passive dye, WENO, stock GM/Redi, an optional immersed bottom, and CPU/GPU controls.

Run in a fresh Julia kernel. `CUDA_LAUNCH_BLOCKING=1` and an explicit synchronization after every timestep make a device exception surface close to the kernel that caused it instead of later in `NaNChecker`. If a GPU case throws `KernelException`, save the complete output and restart the Julia kernel before another GPU run.

In [1]:
ENV["CUDA_LAUNCH_BLOCKING"] = "1"

using Pkg
using Oceananigans
using Oceananigans.Units
using CUDA
using Printf
using Oceananigans.TurbulenceClosures: IsopycnalSkewSymmetricDiffusivity, AdvectiveFormulation

CUDA.allowscalar(false)
println("Julia version: ", VERSION)
Pkg.status(["Oceananigans", "CUDA"])
println("CUDA.functional() = ", CUDA.functional())

[ Info: Precompiling Oceananigans [9e8cae18-63c1-5223-a75c-80ca9d6e9a09](cache misses: mismatched flags (3))
[ Info: Precompiling Oceananigans [9e8cae18-63c1-5223-a75c-80ca9d6e9a09] (cache misses: mismatched flags (6))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
[ Info: Precompiling CUDA [052768ef-5323-5732-b1bb-66c8b64840ba](cache misses: wrong dep version loaded (1), mismatched flags (3))
[ Info: Precompiling CUDA [052768ef-5323-5732-b1bb-66c8b64840ba] (cache misses: wrong dep version loaded (2), mismatched flags (6))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
[ Info: Precompiling JLD2Ext [3dbd0623-ce14-52d8-8601-bc177a3b211d](cache misses: wrong dep version loaded (1), mismatched flags (3))
[ Info: Precompiling JLD2Ext [3dbd0623-ce14-52d8-8601-bc177a3b211d] (cache misses: wrong dep version loaded (2), mismatched flags (6))

SYSTEM: caught exception of type :MethodError 

Julia version: 1.12.6
Status `C:\Users\meghn\OneDrive\Desktop\summer '26 code\ocean modeling\repo-cleanup\ocean-modeling\amazon_river\Project.toml` (empty project)
CUDA.functional() = true


In [2]:
const Nx = 8
const Ny = 8
const Nz = 8
const Lx = 100kilometers
const Ly = 100kilometers
const H = 1000meters
const Δt = 30seconds
const number_of_steps = 10

function build_dye_model(arch; use_redi, immersed)
    z = ExponentialDiscretization(Nz, -H, 0; scale=H/4)
    underlying_grid = RectilinearGrid(
        arch;
        size=(Nx, Ny, Nz),
        halo=(3, 3, 3),
        x=(0, Lx),
        y=(0, Ly),
        z,
        topology=(Bounded, Bounded, Bounded),
    )

    if immersed
        bottom(x, y) = -900meters + 550meters *
                       exp(-((x - Lx/2)^2 + (y - Ly/2)^2) / (20kilometers)^2)
        grid = ImmersedBoundaryGrid(
            underlying_grid, GridFittedBottom(bottom); active_cells_map=true
        )
    else
        grid = underlying_grid
    end

    closure = use_redi ? IsopycnalSkewSymmetricDiffusivity(
        κ_skew=1e3,
        κ_symmetric=1e3,
        skew_flux_formulation=AdvectiveFormulation(),
    ) : nothing

    model = HydrostaticFreeSurfaceModel(
        grid;
        buoyancy=BuoyancyTracer(),
        tracers=(:b, :dye),
        closure,
        momentum_advection=WENOVectorInvariant(order=5),
        tracer_advection=WENO(order=5),
        free_surface=SplitExplicitFreeSurface(grid; substeps=20),
    )

    bᵢ(x, y, z) = 1e-5 * z + 1e-7 * (x - Lx/2)
    dyeᵢ(x, y, z) = exp(-((x - Lx/2)^2 + (y - Ly/2)^2) / (12kilometers)^2) *
                     exp(-((z + 40meters)^2) / (30meters)^2)
    set!(model, b=bᵢ, dye=dyeᵢ)
    return model
end

build_dye_model (generic function with 1 method)

In [3]:
function run_case(name, arch; use_redi, immersed)
    @info "START" name arch use_redi immersed
    model = build_dye_model(arch; use_redi, immersed)
    initial_dye = Array(interior(model.tracers.dye))
    @assert minimum(initial_dye) >= 0
    @assert all(isfinite, initial_dye)

    minimum_dye = minimum(initial_dye)
    maximum_dye = maximum(initial_dye)
    for step in 1:number_of_steps
        time_step!(model, Δt)
        arch isa GPU && CUDA.synchronize()
        dye = Array(interior(model.tracers.dye))
        @assert all(isfinite, dye) "non-finite dye at step $step"
        minimum_dye = min(minimum_dye, minimum(dye))
        maximum_dye = max(maximum_dye, maximum(dye))
        @printf("%s step=%2d dye[min,max]=[% .6e, % .6e]\n",
                name, step, minimum(dye), maximum(dye))
    end
    @info "PASS" name minimum_dye maximum_dye
    return (; name, minimum_dye, maximum_dye)
end

run_case (generic function with 1 method)

## Reduction ladder

Run these cells in order. The first is the CPU reference. The second tests CUDA without GM/Redi. The third adds GM/Redi on a regular grid. The fourth adds the immersed bottom and is the closest small analogue of the Amazon configuration. The first failing case identifies the minimal feature set needed for an issue.

In [4]:
cpu_result = run_case("CPU + GM/Redi + immersed", CPU(); use_redi=true, immersed=true)

┌ Info: START
│   name = "CPU + GM/Redi + immersed"
│   arch = CPU()
│   use_redi = true
└   immersed = true


LoadError: ArgumentError: The grid halo (3, 3, 3) must be at least equal to (5, 5, 4). 
 Note that an ImmersedBoundaryGrid requires an extra halo point in all 
 non-flat directions compared to a non-immersed boundary grid.

In [ ]:
@assert CUDA.functional() "CUDA is unavailable"
gpu_no_redi_result = run_case(
    "GPU + no GM/Redi + immersed", GPU(); use_redi=false, immersed=true
)

In [ ]:
gpu_redi_regular_result = run_case(
    "GPU + GM/Redi + regular", GPU(); use_redi=true, immersed=false
)

In [ ]:
# Suspected case: keep this last because a device exception may require a kernel restart.
gpu_redi_immersed_result = run_case(
    "GPU + GM/Redi + immersed", GPU(); use_redi=true, immersed=true
)

## What to send with an issue

If one small case fails consistently, attach this notebook or the companion `tests/dye_domainerror_reproducer.jl` and include:

1. The first failing case and the cases that passed before it.
2. All output beginning with Julia/Oceananigans/CUDA versions.
3. The complete device exception printed **before** the host `KernelException` stacktrace.
4. Whether the CPU reference passes.
5. Expected: ten timesteps complete with finite dye. Actual: the exact exception and failing step.

If all four cases pass, this is not yet a reproducer. Add one omitted Amazon-model feature at a time; do not file an issue claiming GM/Redi alone causes the exception.